# Perbandingan PCA: zestcore vs Orange vs scikit-learn vs GUI

Dataset: iris (150×4), heart_disease (303×13 → 25 kolom), bank-full (UCI, 45211×16 → 51
kolom, `y` jadi target dan tidak ikut PCA).

- **Acuan**: PCA Orange 3.40.0 lewat projector widget (`Orange.projection.PCA(n_components=100,
  random_state=0)` + preprocessor Continuize → SklImpute → Normalize ON), sama dengan
  setting default widget PCA.
- **Pembanding**: zestcore `pca(X, n_components=100)`; scikit-learn
  `PCA(n_components=min(X.shape), svd_solver="auto")`; bukti GUI di `evidence/pca/`
  (`.npy` komponen, `.pkl` skor, CSV untuk rasio).
- X untuk zestcore dan scikit-learn = X yang sudah diproses preprocessor Orange
  (`preprocessed_X`, ADR 0002).
- Aturan uji dan toleransi: `docs/orange-pca-internals.md`. `np.allclose` default
  (rtol=1e-5, atol=1e-8). Untuk arah varians nol, komponen tidak dibandingkan, sedangkan
  skor dan varians dibandingkan dengan rtol=0, atol=1e-8. Komponen seri dicocokkan dengan
  satu tanda s yang berlaku untuk komponen **dan** skornya.
- Kode pembanding: `tests/pca_compare.py`, `tests/datasets.py`, `tests/evidence_io.py`
  (sama dengan yang dipakai pytest).


In [1]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
import Orange
from sklearn.decomposition import PCA

print("Python      ", sys.version.split()[0])
print("Orange      ", Orange.__version__)
print("NumPy       ", np.__version__)
print("scikit-learn", sklearn.__version__)

# Notebook dijalankan dari notebooks/; semua path relatif ke root repo.
ROOT = Path.cwd().parent
assert (ROOT / "pyproject.toml").exists(), "jalankan notebook dari folder notebooks/"
sys.path[:0] = [str(ROOT), str(ROOT / "tests")]

# DeprecationWarning dari orangecanvas (paket Orange) saat import OWPCA; bukan dari kode ini.
warnings.filterwarnings("ignore", category=DeprecationWarning, module="orangecanvas")

from datasets import load_table
from evidence_io import EVIDENCE, read_components_csv, read_components_npy, read_scores_pkl
from pca_compare import (compare, compare_ratio, compare_variance, orange_widget_pca,
                         preprocessed_X, tied_rows, zero_variance_count)
from zestcore.pca import pca

Python       3.12.12
Orange       3.40.0
NumPy        2.1.3
scikit-learn 1.5.2


**Data dan keempat hasil PCA.** Bukti GUI dibaca hanya bila ketiga file (`_components.npy`,
`_components.csv`, `_scores.pkl`) ada. Kalau belum ada, barisnya ditulis "menyusul". Sel ini
juga memeriksa bahwa jumlah arah varians nol dari rumus (aturan uji 2) sama dengan jumlah
`explained_variance_ < 1e-10` yang terukur di Orange.


In [2]:
CASES = [("iris", "iris_pca_norm-on_4pc"),
         ("heart_disease", "heart_pca_norm-on_25pc"),
         ("bank-full", "bank_pca_norm-on_51pc")]
GUI_FILES = ("_components.npy", "_components.csv", "_scores.pkl")

HASIL = {}
for dataset, prefix in CASES:
    table = load_table(dataset)
    X = preprocessed_X(table)
    proj, scores = orange_widget_pca(table)
    z = pca(X, n_components=100)
    sk = PCA(n_components=min(X.shape), svd_solver="auto").fit(X)
    gui = None
    if all((EVIDENCE / f"{prefix}{s}").exists() for s in GUI_FILES):
        _, gui_ratio = read_components_csv(f"{prefix}_components.csv")
        gui = {"comp": read_components_npy(f"{prefix}_components.npy"),
               "scores": read_scores_pkl(f"{prefix}_scores.pkl"), "ratio_csv": gui_ratio}
    n_null = zero_variance_count(table)
    n_null_terukur = int((proj.explained_variance_ < 1e-10).sum())
    assert n_null == n_null_terukur, (dataset, n_null, n_null_terukur)
    HASIL[dataset] = {"X": X, "proj": proj, "scores": scores, "z": z, "sk": sk,
                      "gui": gui, "n_null": n_null}
    print(f"{dataset:14s} X {X.shape[0]}x{X.shape[1]}, k = {len(proj.components_)}, "
          f"solver sklearn = {sk._fit_svd_solver}, arah nol rumus = terukur = {n_null}, "
          f"bukti GUI: {'ada' if gui else 'menyusul'}")

iris           X 150x4, k = 4, solver sklearn = covariance_eigh, arah nol rumus = terukur = 0, bukti GUI: ada
heart_disease  X 303x25, k = 25, solver sklearn = covariance_eigh, arah nol rumus = terukur = 7, bukti GUI: ada


bank-full      X 45211x51, k = 51, solver sklearn = covariance_eigh, arah nol rumus = terukur = 9, bukti GUI: ada


**Fungsi ringkasan.** `compare` dan `compare_variance` (dari `tests/pca_compare.py`)
mengeluarkan satu baris per PC. Fungsi di bawah hanya mengambil selisih maksimum per
metrik, tanpa aturan uji baru. Kolom "skor nol" berisi skor di ruang nol; komponen ruang nol
tidak dibandingkan (aturan uji 2).


In [3]:
METRIK = ["komponen", "skor", "skor nol", "expl. var", "rasio"]


def ringkas(dataset, pembanding, comp, scores, ev_rows, rasio_rows, catatan=""):
    h = HASIL[dataset]
    rows = compare(h["proj"].components_, h["scores"], comp, scores, h["n_null"])
    k = len(h["proj"].components_) - h["n_null"]
    grup = {"komponen": [r for r in rows if r["jenis"] == "komponen"],
            "skor": [r for r in rows if r["jenis"] == "skor" and r["pc"] <= k],
            "skor nol": [r for r in rows if r["jenis"] == "skor" and r["pc"] > k],
            "expl. var": ev_rows, "rasio": rasio_rows}
    hasil = {"dataset": dataset, "pembanding": pembanding}
    for m in METRIK:
        hasil[m] = max(r["maks"] for r in grup[m]) if grup[m] else "–"
    if not ev_rows:
        hasil["expl. var"] = "– (GUI tidak mengeluarkan)"
    hasil["seri (s)"] = ", ".join(f"PC{r['pc']} ({r['s']:+d})"
                                  for r in grup["komponen"] if r["seri"]) or "–"
    hasil["allclose"] = all(r["allclose"] for g in grup.values() for r in g)
    hasil["catatan"] = catatan
    return hasil


def menyusul(dataset, pembanding):
    return {"dataset": dataset, "pembanding": pembanding, **{m: "menyusul" for m in METRIK},
            "seri (s)": "menyusul", "allclose": None, "catatan": "bukti GUI belum ada"}

**1. zestcore vs Orange** (inti NumPy murni vs acuan).

In [4]:
tabel_zest = []
for dataset, h in HASIL.items():
    z, proj = h["z"], h["proj"]
    tabel_zest.append(ringkas(
        dataset, "zestcore vs Orange", z.components, z.scores,
        compare_variance(proj.explained_variance_, z.explained_variance, h["n_null"]),
        compare_variance(proj.explained_variance_ratio_, z.explained_variance_ratio, h["n_null"])))
pd.DataFrame(tabel_zest)

,dataset,pembanding,komponen,skor,skor nol,expl. var,rasio,seri (s),allclose,catatan
0,iris,zestcore vs Orange,0.0,5.551115e-17,–,0.0,0.0,–,True,
1,heart_disease,zestcore vs Orange,0.0,8.881784e-16,0.0,0.0,0.0,"PC2 (+1), PC3 (+1)",True,
2,bank-full,zestcore vs Orange,0.0,3.108624e-15,0.0,0.0,0.0,"PC2 (+1), PC4 (+1), PC6 (+1), PC7 (+1)",True,


**2. scikit-learn vs Orange.** Orange memanggil sklearn yang sama pada X yang sama, jadi
selisihnya diharapkan 0.0. Solver yang dipilih `auto` dicetak di sel data (`covariance_eigh`
untuk ketiga dataset).


In [5]:
tabel_sk = []
for dataset, h in HASIL.items():
    sk, proj = h["sk"], h["proj"]
    tabel_sk.append(ringkas(
        dataset, "scikit-learn vs Orange", sk.components_, sk.transform(h["X"]),
        compare_variance(proj.explained_variance_, sk.explained_variance_, h["n_null"]),
        compare_variance(proj.explained_variance_ratio_, sk.explained_variance_ratio_, h["n_null"])))
pd.DataFrame(tabel_sk)

,dataset,pembanding,komponen,skor,skor nol,expl. var,rasio,seri (s),allclose,catatan
0,iris,scikit-learn vs Orange,0.0,0.000000e+00,–,0.0,0.0,–,True,
1,heart_disease,scikit-learn vs Orange,0.0,0.000000e+00,0.0,0.0,0.0,"PC2 (+1), PC3 (+1)",True,
2,bank-full,scikit-learn vs Orange,0.0,1.776357e-15,0.0,0.0,0.0,"PC2 (+1), PC4 (+1), PC6 (+1), PC7 (+1)",True,


**3. GUI vs Orange.** Komponen dari `.npy` dan skor dari `.pkl` (presisi penuh). GUI hanya
mengeluarkan rasio varians, lewat CSV Components yang ditulis `%g`, jadi untuk PC non-nol
rasio dibandingkan dengan rtol=1e-5, atol=0 (`compare_ratio`).


In [6]:
tabel_gui = []
for dataset, h in HASIL.items():
    g = h["gui"]
    if g is None:
        tabel_gui.append(menyusul(dataset, "GUI vs Orange"))
        continue
    tabel_gui.append(ringkas(
        dataset, "GUI vs Orange", g["comp"], g["scores"], [],
        compare_ratio(g["ratio_csv"], h["proj"].explained_variance_ratio_, h["n_null"]),
        catatan="rasio dari CSV (%g)"))
pd.DataFrame(tabel_gui)

,dataset,pembanding,komponen,skor,skor nol,expl. var,rasio,seri (s),allclose,catatan
0,iris,GUI vs Orange,0.0,0.0,–,– (GUI tidak mengeluarkan),4.790620e-07,–,True,rasio dari CSV (%g)
1,heart_disease,GUI vs Orange,0.0,0.0,0.0,– (GUI tidak mengeluarkan),3.584042e-07,"PC2 (+1), PC3 (+1)",True,rasio dari CSV (%g)
2,bank-full,GUI vs Orange,0.0,0.0,0.0,– (GUI tidak mengeluarkan),4.846691e-08,"PC2 (+1), PC4 (+1), PC6 (+1), PC7 (+1)",True,rasio dari CSV (%g)


**Tabel selisih maksimum** (dataset × pembanding × metrik, acuan = Orange). Angka = selisih
absolut maksimum, dan `allclose` = semua baris PC lolos aturan uji.


In [7]:
def fmt(v):
    return f"{v:.2e}" if isinstance(v, float) else v


summary = pd.DataFrame(tabel_zest + tabel_sk + tabel_gui)
summary = summary.sort_values("dataset", key=lambda s: s.map(list(HASIL).index), kind="stable")
summary[METRIK] = summary[METRIK].map(fmt)
summary.reset_index(drop=True)

,dataset,pembanding,komponen,skor,skor nol,expl. var,rasio,seri (s),allclose,catatan
0,iris,zestcore vs Orange,0.00e+00,5.55e-17,–,0.00e+00,0.00e+00,–,True,
1,iris,scikit-learn vs Orange,0.00e+00,0.00e+00,–,0.00e+00,0.00e+00,–,True,
2,iris,GUI vs Orange,0.00e+00,0.00e+00,–,– (GUI tidak mengeluarkan),4.79e-07,–,True,rasio dari CSV (%g)
3,heart_disease,zestcore vs Orange,0.00e+00,8.88e-16,4.44e-16,0.00e+00,0.00e+00,"PC2 (+1), PC3 (+1)",True,
4,heart_disease,scikit-learn vs Orange,0.00e+00,0.00e+00,0.00e+00,0.00e+00,0.00e+00,"PC2 (+1), PC3 (+1)",True,
5,heart_disease,GUI vs Orange,0.00e+00,0.00e+00,0.00e+00,– (GUI tidak mengeluarkan),3.58e-07,"PC2 (+1), PC3 (+1)",True,rasio dari CSV (%g)
6,bank-full,zestcore vs Orange,0.00e+00,3.11e-15,1.33e-15,0.00e+00,0.00e+00,"PC2 (+1), PC4 (+1), PC6 (+1), PC7 (+1)",True,
7,bank-full,scikit-learn vs Orange,0.00e+00,1.78e-15,8.88e-16,0.00e+00,0.00e+00,"PC2 (+1), PC4 (+1), PC6 (+1), PC7 (+1)",True,
8,bank-full,GUI vs Orange,0.00e+00,0.00e+00,0.00e+00,– (GUI tidak mengeluarkan),4.85e-08,"PC2 (+1), PC4 (+1), PC6 (+1), PC7 (+1)",True,rasio dari CSV (%g)


**Komponen seri dan arah varians nol** (dari acuan Orange). Komponen seri: elemen
|terbesar| dan |kedua terbesar| berjarak < 1e-10, sehingga tanda komponen ditentukan oleh
galat pembulatan. Arah nol: eigenvalue < 1e-10; vektornya tidak unik.


In [8]:
info = []
for dataset, h in HASIL.items():
    k_all = len(h["proj"].components_)
    k = k_all - h["n_null"]
    info.append({"dataset": dataset, "X": f"{h['X'].shape[0]}x{h['X'].shape[1]}",
                 "k": k_all, "solver": h["sk"]._fit_svd_solver,
                 "komponen seri": ", ".join(f"PC{i + 1}"
                                            for i in tied_rows(h["proj"].components_[:k])) or "–",
                 "arah nol (rumus = terukur)": h["n_null"],
                 "PC ruang nol": f"PC{k + 1}-{k_all}" if h["n_null"] else "–"})
pd.DataFrame(info)

,dataset,X,k,solver,komponen seri,arah nol (rumus = terukur),PC ruang nol
0,iris,150x4,4,covariance_eigh,–,0,–
1,heart_disease,303x25,25,covariance_eigh,"PC2, PC3",7,PC19-25
2,bank-full,45211x51,51,covariance_eigh,"PC2, PC4, PC6, PC7",9,PC43-51


**Pemeriksaan** (sel ini gagal bila ada perbandingan yang tidak lolos aturan uji).

In [9]:
diuji = pd.DataFrame(tabel_zest + tabel_sk + tabel_gui).dropna(subset=["allclose"])
assert diuji["allclose"].astype(bool).all(), diuji[~diuji["allclose"].astype(bool)]
print(f"{len(diuji)} perbandingan lolos, "
      f"{sum(r['allclose'] is None for r in tabel_gui)} bukti GUI menyusul")

9 perbandingan lolos, 0 bukti GUI menyusul


**Sisa selisih: skor.** Komponen, explained variance, dan rasio zestcore sama persis
dengan sklearn/Orange (0.0), karena kovarians dihitung dengan bentuk dan urutan perkalian
yang sama dengan sklearn (`_pca.py:611-617`). Skor masih berbeda di orde ≤1e-14, walaupun
rumusnya sama (X·Cᵀ − mean·Cᵀ, `_base.py:149, 156`). Penyebabnya tata letak memori:
`components_` sklearn disimpan F-order, milik zestcore C-order, sehingga BLAS mengambil
jalur perkalian yang berbeda untuk `X @ components.T`. Ini detail implementasi BLAS, bukan
perbedaan metode, jadi zestcore tidak diubah untuk menirunya.

Sel di bawah menghitung skor zestcore dua kali dengan rumus yang sama: dengan komponen
zestcore apa adanya (C-order) dan dengan salinan F-order. Keduanya dibandingkan dengan
`sk.transform(X)`. Kolom terakhir mengukur selisih skor Orange sendiri (`model(table)`) vs
`sk.transform(X)` pada X yang sama.


In [10]:
bukti = []
for dataset, h in HASIL.items():
    X, sk, z = h["X"], h["sk"], h["z"]
    comp_f = np.asfortranarray(z.components)
    skor_f = X @ comp_f.T - z.mean.reshape(1, -1) @ comp_f.T
    bukti.append({"dataset": dataset,
                  "komponen zestcore vs sklearn": np.abs(z.components - sk.components_).max(),
                  "expl. var zestcore vs sklearn": np.abs(z.explained_variance
                                                          - sk.explained_variance_).max(),
                  "skor zestcore (C-order) vs sklearn": np.abs(z.scores - sk.transform(X)).max(),
                  "skor F-order vs sklearn": np.abs(skor_f - sk.transform(X)).max(),
                  "skor Orange vs sklearn": np.abs(h["scores"] - sk.transform(X)).max()})
pd.DataFrame(bukti).map(fmt)

,dataset,komponen zestcore vs sklearn,expl. var zestcore vs sklearn,skor zestcore (C-order) vs sklearn,skor F-order vs sklearn,skor Orange vs sklearn
0,iris,0.00e+00,0.00e+00,5.55e-17,0.00e+00,0.00e+00
1,heart_disease,0.00e+00,0.00e+00,8.88e-16,0.00e+00,0.00e+00
2,bank-full,0.00e+00,0.00e+00,1.78e-15,0.00e+00,1.78e-15


**Pertanyaan terbuka.** Di bank-full, skor dari model Orange (`model(table)`, jalur
`Table.transform`) berbeda 1.8e-15 dari `sk.transform(X)` pada X yang sama, padahal Orange
memanggil sklearn (`Orange/projection/base.py:115-116`). Di iris dan heart_disease
selisihnya 0.0. Penyebabnya di jalur transform Orange belum diselidiki. Selisih ini jauh
di bawah toleransi, sehingga tidak mengubah kesimpulan.


## bank-full: kesimpulan

**Identik.** Komponen, explained variance, dan rasio sama persis dengan Orange (selisih
0.0); skor identik dalam toleransi aturan uji (selisih maks 3.1e-15). Alasannya:

- bank-full (45211×51 setelah Continuize) memenuhi syarat jalur `covariance_eigh`
  (p = 51 ≤ 1000 dan n ≥ 10p = 510), sehingga Orange dan zestcore menempuh langkah yang
  sama: kovarians ddof=1 dengan bentuk dan urutan perkalian sklearn, eigen-dekomposisi,
  `svd_flip` berbasis baris, dan skor X·Cᵀ − mean·Cᵀ.
- Selisih skor berasal dari dua hal yang sama-sama bukan perbedaan metode: tata letak
  memori komponen saat perkalian BLAS (1.8e-15, sel bukti) dan jalur transform Orange
  sendiri (1.8e-15, pertanyaan terbuka). Keduanya jauh di bawah toleransi (rtol=1e-5,
  atol=1e-8).
- 9 arah varians nol (PC43-51), sama dengan prediksi rumus aturan uji 2. Vektor ruang
  nol tidak unik, jadi komponennya tidak dibandingkan; skornya (maks 1.3e-15) lolos
  atol=1e-8.
- 4 komponen seri (PC2, 4, 6, 7) berasal dari variabel biner yang di-Continuize jadi `x` dan
  `1-x`. Tandanya ditentukan galat pembulatan, sehingga dicocokkan dengan satu tanda s
  untuk komponen dan skornya.
- scikit-learn vs Orange: komponen dan varians 0.0, skor 1.8e-15 (jalur transform Orange,
  lihat pertanyaan terbuka).
- Bukti GUI bank-full (`evidence/pca/bank_pca_norm-on_51pc_*`): komponen `.npy` dan skor
  `.pkl` sama persis dengan Orange lewat skrip (selisih 0.0); rasio dari CSV berbeda maks
  4.9e-8, sesuai pembulatan `%g`. Baris "GUI vs Orange" di tabel dibaca otomatis dari
  file tersebut.


## Narasi untuk BAB IV

PCA NumPy murni di `zestcore` dibandingkan dengan PCA bawaan Orange 3.40.0 pada tiga
dataset: iris (150×4), heart_disease (303×25 setelah Continuize), dan bank-full (45211×51).
Setting mengikuti widget PCA default (Normalize ON, k = min(n, p, 100)), dan X untuk
zestcore adalah keluaran preprocessor Orange (Continuize, imputasi mean, Normalize).
Acuannya adalah PCA Orange lewat skrip. Pembandingnya zestcore, scikit-learn 1.5.2, dan
keluaran GUI Orange yang direkam di `evidence/pca/`. Kriteria identik adalah `np.allclose`
dengan rtol=1e-5 dan atol=1e-8, dengan selisih maksimum dilaporkan.

Ketiga pembanding lolos di ketiga dataset. Komponen, explained
variance, dan rasio varians zestcore sama persis dengan Orange (selisih 0.0) di ketiga
dataset. Skor zestcore berbeda paling besar 3.1e-15 (bank-full). Komponen dan skor
scikit-learn serta GUI sama persis dengan Orange, kecuali skor scikit-learn pada
bank-full (1.8e-15). Rasio varians GUI hanya tersedia dari CSV yang dibulatkan ke 6 digit
signifikan, sehingga selisihnya (maks 4.8e-7) adalah pembulatan tersebut.

Ada dua hal yang perlu diperhatikan saat membandingkan. Pertama, komponen yang dibentuk
dari variabel biner bisa *seri*: dua loading sama besar dengan tanda berlawanan, sehingga
tanda komponen ditentukan galat pembulatan (heart_disease PC2, PC3; bank-full PC2, 4, 6, 7).
Komponen seperti ini dibandingkan setelah tandanya disamakan, dan tanda yang sama dipakai
untuk skornya. Kedua, setiap variabel diskrit menghasilkan satu arah varians nol (7 di
heart_disease, 9 di bank-full, sesuai prediksi rumus). Vektor di arah ini tidak unik, jadi
yang dibandingkan hanya skornya.

Selisih skor yang tersisa tidak berasal dari perbedaan metode. Rumus skor zestcore sama
dengan scikit-learn, tetapi matriks komponen disimpan dengan tata letak memori yang
berbeda (C-order vs F-order), sehingga pustaka BLAS mengambil jalur perkalian yang
berbeda. Dengan salinan komponen F-order, skor zestcore sama persis dengan scikit-learn.
Selisih skor scikit-learn vs Orange pada bank-full (1.8e-15) berasal dari jalur transform
Orange sendiri dan belum diselidiki. Semua selisih ini berada di orde galat pembulatan
floating-point dan jauh di bawah toleransi.
